# Setup

In [17]:
import sys
from pathlib import Path

# Make the audit module importable from the notebook
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)
np.random.seed(42)

RUN_ID = 26


In [19]:
from app.repository.backtest.database import SessionLocal
from app.repository.backtest.models import Run, RunConfig

db = SessionLocal()
recent = (
    db.query(Run.id, Run.status, Run.created_at, RunConfig.symbol, RunConfig.timeframe)
    .join(RunConfig, RunConfig.run_id == Run.id)
    .filter(Run.status == "completed")
    .order_by(Run.created_at.desc())
    .limit(30)
    .all()
)
db.close()

import pandas as pd
pd.DataFrame(recent, columns=["run_id", "status", "created_at", "symbol", "timeframe"])

,run_id,status,created_at,symbol,timeframe
0,27,completed,2026-04-29 14:20:37.838333,BATCH,15m
1,26,completed,2026-04-29 14:02:45.574892,PORTFOLIO,15m
2,25,completed,2026-04-29 14:02:39.649005,PORTFOLIO,1h
3,24,completed,2026-04-29 13:56:08.516459,PORTFOLIO,1h
4,23,completed,2026-04-17 10:15:50.583941,BTC/USDT,15m
5,22,completed,2026-04-17 08:01:35.786782,BTC/USDT,15m
6,21,completed,2026-04-17 07:51:50.046907,BTC/USDT,15m
7,20,completed,2026-04-17 06:35:19.344229,BTC/USDT,15m
8,19,completed,2026-04-17 06:35:12.204351,BTC/USDT,15m
9,17,completed,2026-04-15 17:03:54.405152,BTC/USDT,15m


# Trade log + sanity dashboard

In [ ]:
from app.backtest.audit.trade_log import build_trade_log
from app.backtest.audit.sanity import run_sanity_audits

RUN_ID = 27
tl = build_trade_log(RUN_ID)
print(f"Run {tl.run_id}: {len(tl.df)} closed trades, {tl.dropped_open_count} open dropped\n")

sanity = run_sanity_audits(tl, single_direction=True)
print(f"Sanity verdict: {'PASS' if sanity['passed'] else 'FAIL'}\n")

CHECK_KEYS = ("pnl_concentration", "long_short_symmetry", "cost_sensitivity")
for name in CHECK_KEYS:
    result = sanity[name]
    icon = '✓' if result.get('passed', False) else ('—' if not result.get('applicable', True) else '✗')
    val = result.get('value', 'n/a')
    print(f"  {icon} {name}: value={val}")

tl.df.head()

Run 26: 138 closed trades, 0 open dropped

Sanity verdict: FAIL

  ✓ pnl_concentration: value=0.09174729986526595
  ✓ long_short_symmetry: value=n/a
  ✗ cost_sensitivity: value=n/a


,entry_time,exit_time,side,symbol,entry_price,exit_price,qty,ret_pct,ret_abs,holding_hours,exit_reason,run_id
0,2026-01-31 17:45:00,2026-01-31 19:45:00,BUY,AXS/USDT,1.9190,1.8780,4166.6667,-2.3344,-186.6542,2.0000,CLOSE_BY_CANDLE_SL,26
1,2026-02-01 03:45:00,2026-02-01 05:15:00,BUY,HBAR/USDT,0.0902,0.0909,35842.2939,0.6188,20.0000,1.5000,SL,26
2,2026-02-01 04:15:00,2026-02-01 08:15:00,BUY,ETHFI/USDT,0.4849,0.4920,3246.7532,1.2704,20.0000,4.0000,SL,26
3,2026-02-01 04:30:00,2026-02-01 18:30:00,BUY,XLM/USDT,0.1768,0.1801,19762.8458,1.6357,57.1664,14.0000,TP1,26
4,2026-02-01 05:30:00,2026-02-01 21:30:00,BUY,PYTH/USDT,0.0515,0.0527,52631.5789,2.0727,56.2258,16.0000,TP1,26


# Equity curve

In [ ]:
equity = tl.df.set_index('exit_time')['ret_abs'].cumsum()
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(equity.index, equity.values, color='steelblue', linewidth=2)
ax.fill_between(equity.index, equity.values, 0,
                where=(equity.values >= 0), color='green', alpha=0.2)
ax.fill_between(equity.index, equity.values, 0,
                where=(equity.values < 0), color='red', alpha=0.2)
ax.axhline(0, color='black', linestyle='--', linewidth=0.5)
ax.set_title(f'Run {RUN_ID} — Cumulative PnL ($)')
ax.set_ylabel('Cumulative $')
plt.tight_layout()
plt.show()

# PnL concentration

In [ ]:
sorted_abs = tl.df['ret_abs'].abs().sort_values(ascending=False).reset_index(drop=True)
cum_share = sorted_abs.cumsum() / sorted_abs.sum()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(len(sorted_abs)), sorted_abs, color='steelblue', alpha=0.7)
ax.set_xlabel('Trade rank (by |PnL|)')
ax.set_ylabel('|PnL| ($)', color='steelblue')

ax2 = ax.twinx()
ax2.plot(range(len(cum_share)), cum_share, color='crimson', linewidth=2)
ax2.axhline(0.5, color='crimson', linestyle='--', alpha=0.5, label='50% threshold')
ax2.axvline(4, color='gray', linestyle=':', alpha=0.5, label='Top 5')
ax2.set_ylabel('Cumulative share', color='crimson')
ax2.legend(loc='center right')
ax2.set_ylim(0, 1.05)

top5_share = sanity['pnl_concentration']['value']
plt.title(f'Run {RUN_ID} — PnL concentration (top 5 = {top5_share:.2%})')
plt.tight_layout()
plt.show()

# Signal panel + RSI distribution

In [ ]:
from app.backtest.audit.signal_panel import build_signal_panel

panel = build_signal_panel(SYMBOL, TIMEFRAME)
print(f"Panel: {len(panel.df)} bars, {panel.symbol} {panel.timeframe}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(panel.df['rsi_14'], bins=50, color='steelblue', alpha=0.7)
axes[0].axvline(30, color='green', linestyle='--', alpha=0.5, label='30 oversold')
axes[0].axvline(70, color='red', linestyle='--', alpha=0.5, label='70 overbought')
axes[0].set_title('RSI(14) distribution')
axes[0].set_xlabel('RSI')
axes[0].legend()

axes[1].hist(panel.df['fwd_logret_4'], bins=50, color='crimson', alpha=0.7)
axes[1].axvline(0, color='black', linestyle='--', alpha=0.5)
axes[1].set_title('Forward log return (h=4 bars = 1h)')
axes[1].set_xlabel('Log return')
plt.tight_layout()
plt.show()

# IC preview (decile chart, the moment of truth)

In [ ]:
# Decile analysis — bin RSI, plot mean forward return per bin
# Monotonic = real signal. Zigzag = noise.

from scipy.stats import spearmanr

panel.df['rsi_decile'] = pd.qcut(panel.df['rsi_14'], 10, labels=False, duplicates='drop')
decile_summary = panel.df.groupby('rsi_decile').agg({
    'rsi_14': 'mean',
    'fwd_logret_4': ['mean', 'std', 'count']
})

# Spearman IC
ic, p_val = spearmanr(panel.df['rsi_14'], panel.df['fwd_logret_4'])
print(f"Spearman IC (h=4 bars): {ic:.4f}, p-value: {p_val:.4e}")
print(f"Sample size: {len(panel.df):,} bars")

fig, ax = plt.subplots(figsize=(12, 5))
deciles = decile_summary.index.values
mean_returns = decile_summary[('fwd_logret_4', 'mean')].values * 100  # to bps-ish
colors = ['red' if r < 0 else 'green' for r in mean_returns]
ax.bar(deciles, mean_returns, color=colors, alpha=0.7)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('RSI decile (0=lowest, 9=highest)')
ax.set_ylabel('Mean forward log return × 100 (h=4 bars)')
ax.set_title(f'RSI decile vs forward return — IC={ic:.4f}, p={p_val:.2e}')
plt.tight_layout()
plt.show()